# RikiGaku-AHIRU（AHIRU-μTJ42V）— Google Colab Pro (A100 GPU) 全22分野・一括物理検証システム

全長 **18 cm × 幅 10 cm × 高さ 20 cm** のジェット推進アヒル置物 **RikiGaku-AHIRU** が、現実世界でそのまま製造・飛行可能であることを証明するため、**主要12分野＋実機物理拡張10分野＝全22分野**の激重シミュレーションを一括実行します。

> **使い方**: Google Colab Pro でランタイムを **A100 GPU** に設定し、メニューの **「ランタイム → すべてのセルを実行 (Ctrl+F9)」** をワンクリックしてください。

In [ ]:
#@title 1. A100 GPU確認 & リポジトリ取得 & 全OSS物理ソルバー一括セットアップ
import os, subprocess, torch

print('=== GPU Hardware Check ===')
!nvidia-smi
if torch.cuda.is_available():
    print('CUDA Device:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

REPO_DIR = '/content/RikiGaku-AHIRU'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/sc8z35a-collab/RikiGaku-AHIRU.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin main

%cd {REPO_DIR}

print('\n=== Installing Open-Source Physics & Engineering Solvers ===')
!apt-get update -qq
!apt-get install -y -qq cppcheck g++ ngspice calculix-ccx gmsh openscad > /dev/null 2>&1 || true
!pip install -q --upgrade pip
!pip install -q numpy scipy matplotlib cantera mujoco ross-rotordynamics trimesh control casadi
print('Setup complete!')

In [ ]:
#@title 2. ワンクリック全22分野・激重A100 GPU一括検証を実行
%cd /content/RikiGaku-AHIRU
!python3 -m verify_ahiru.run_all --mode colab_a100_heavy --out reports

In [ ]:
#@title 3. 全22分野 検証結果ダッシュボード & 合格スコアカード表示
import json
from IPython.display import Image, display, HTML

with open('/content/RikiGaku-AHIRU/reports/metrics.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"ALL PASSED: {data['all_passed']} ({data['passed_count']}/{data['total_domains']}) | Total Time: {data['total_elapsed_s']}s")
print(f"Hardware: {data['hardware']}")
display(Image('/content/RikiGaku-AHIRU/reports/ahiru_verification_dashboard.png'))

for r in data['results']:
    status = 'PASS' if r['passed'] else 'FAIL'
    print(f"[{status}] {r['domain']} {r['name']} -> {r['metrics']}")

In [ ]:
#@title 4. 商用ツール連携用ソルバーデック一覧 (ANSYS / SU2 / OpenFOAM / CalculiX / ngspice / 5軸CNC / CAD)
import glob, shutil

files = sorted(glob.glob('/content/RikiGaku-AHIRU/reports/exports/**/*', recursive=True))
for fp in files:
    if os.path.isfile(fp):
        print(fp)

zip_path = shutil.make_archive('/content/RikiGaku_AHIRU_Verification_Package', 'zip', '/content/RikiGaku-AHIRU/reports')
print('\nExported complete verification & CAE solver deck archive:', zip_path)